# 10_final_evaluate
## Comprehensive Feature Selection Evaluation - All Methods

This notebook evaluates and compares:
- **Baseline**: Raw data with all original features
- **Filter Methods**: Variance, ANOVA, Chi-Squared, Correlation, Mutual Information
- **Wrapper Methods (Raw)**: Sklearn SFS and Seeded SFS on raw features
- **Wrapper Methods (Union)**: Sklearn SFS and Seeded SFS on union features

Shows accuracy metrics and percentage improvement over baseline for all methods combined.

### Setup & Configuration

In [ ]:
import sys
sys.path.insert(0, '/home/andev/Dev/pka/feature_selection/wrapper-w-filter')

from pathlib import Path
from src.modeling.evaluation import ModelEvaluator
from src.config import ProjectPath

### Dataset Configuration

In [ ]:
data_name = "adenocarcinoma"
n_features = 50
n_splits = 5

path = ProjectPath(data_name, n_features)

print(f"Dataset: {data_name}")
print(f"Features: {n_features}")
print(f"Evaluation dir: {path.evaluation_dir}")

### Initialize Evaluator

In [ ]:
evaluator = ModelEvaluator(data_name, custom_base_dir=path.evaluation_dir)

### 1. Evaluate Baseline (All Features)

In [ ]:
print("\n" + "-"*80)
print("PHASE 1: BASELINE EVALUATION (All Original Features)")
print("-"*80)

evaluator.evaluate_baseline(str(path.raw_path), n_splits=n_splits)

### 2. Evaluate Filter Methods

In [ ]:
print("\n" + "-"*80)
print("PHASE 2: FILTER METHODS EVALUATION")
print("-"*80)

evaluator.evaluate_filtered_features(str(path.filter_dir), n_splits=n_splits)

### 3. Evaluate Wrapper Methods - Raw Variant

In [ ]:
print("\n" + "-"*80)
print("PHASE 3A: WRAPPER METHODS EVALUATION (Raw Variant)")
print("-"*80)

# Sklearn SFS - Raw
sk_scoring = "accuracy"
sk_max_features = "auto"
sk_cv = 5
sk_algorithm_name = "sklearnsfsselector"
sk_model = "logisticregression"
path_sklearn_raw = path.wrapper_file(
    data_variant="raw",
    algorithm_name=sk_algorithm_name,
    suffix=f"{sk_model}_{sk_scoring}_{sk_max_features}max_{sk_cv}cv_raw"
)

print(f"\nSklearn SFS (Raw) path: {path_sklearn_raw}")
if path_sklearn_raw.exists():
    evaluator.evaluate_custom_file(str(path_sklearn_raw), "Sklearn_SFS_Raw", n_splits=n_splits)
else:
    print(f"WARNING: File not found: {path_sklearn_raw}")

In [ ]:
# Seeded SFS - Raw
scoring = "accuracy"
max_features = 20
patience = 5
seeds = 1
cv = 5
algorithm_name = "seededsfsselector"

path_seeded_raw = path.wrapper_file(
    data_variant="raw",
    algorithm_name=algorithm_name,
    suffix=f"log_{scoring}_{max_features}max_{seeds}seeds_{patience}pat_{cv}cv_raw"
)

print(f"\nSeeded SFS (Raw) path: {path_seeded_raw}")
if path_seeded_raw.exists():
    evaluator.evaluate_custom_file(str(path_seeded_raw), "Seeded_SFS_Raw", n_splits=n_splits)
else:
    print(f"WARNING: File not found: {path_seeded_raw}")

### 4. Evaluate Wrapper Methods - Union Variant

In [ ]:
print("\n" + "-"*80)
print("PHASE 3B: WRAPPER METHODS EVALUATION (Union Variant)")
print("-"*80)

# Sklearn SFS - Union
path_sklearn_union = path.wrapper_file(
    data_variant="union",
    algorithm_name=sk_algorithm_name,
    suffix=f"{sk_model}_{sk_scoring}_{sk_max_features}max_{sk_cv}cv_union"
)

print(f"\nSklearn SFS (Union) path: {path_sklearn_union}")
if path_sklearn_union.exists():
    evaluator.evaluate_custom_file(str(path_sklearn_union), "Sklearn_SFS_Union", n_splits=n_splits)
else:
    print(f"WARNING: File not found: {path_sklearn_union}")

In [ ]:
# Seeded SFS - Union
path_seeded_union = path.wrapper_file(
    data_variant="union",
    algorithm_name=algorithm_name,
    suffix=f"log_{scoring}_{max_features}max_{seeds}seeds_{patience}pat_{cv}cv_union"
)

print(f"\nSeeded SFS (Union) path: {path_seeded_union}")
if path_seeded_union.exists():
    evaluator.evaluate_custom_file(str(path_seeded_union), "Seeded_SFS_Union", n_splits=n_splits)
else:
    print(f"WARNING: File not found: {path_seeded_union}")

### 5. Generate Comprehensive Report & Visualization

In [ ]:
print("\n" + "-"*80)
print("PHASE 4: GENERATING COMPREHENSIVE REPORT")
print("-"*80)

result_df = evaluator.generate_report_and_plot(
    experiment_prefix=f"final_evaluation_all_methods_{data_name}",
    chart_title=f"Comprehensive Feature Selection Evaluation - All Methods"
)

### 6. Summary Table with Improvement Over Baseline

In [ ]:
if result_df is not None:
    # Group results by method and model to get mean accuracy
    summary = result_df.groupby(['Method', 'Model']).agg({
        'Acc': ['mean', 'std', 'count']
    }).round(4)
    summary.columns = ['Mean_Accuracy', 'Std_Accuracy', 'N_Folds']
    summary = summary.reset_index()
    
    # Find baseline accuracy (Method='None')
    baseline_acc = summary[summary['Method'] == 'None']['Mean_Accuracy'].values
    
    if len(baseline_acc) > 0:
        baseline_mean = baseline_acc[0]
        print(f"\nBASELINE ACCURACY (All Features): {baseline_mean:.4f}")
        print("\n" + "="*100)
        print("IMPROVEMENT OVER BASELINE")
        print("="*100)
        
        summary['Improvement'] = summary['Mean_Accuracy'] - baseline_mean
        summary['Improvement_%'] = (summary['Improvement'] / baseline_mean * 100).round(2)
        
        # Sort by improvement
        summary_sorted = summary.sort_values('Improvement_%', ascending=False)
        
        # Display
        display_df = summary_sorted[['Method', 'Model', 'Mean_Accuracy', 'Std_Accuracy', 'Improvement', 'Improvement_%']].copy()
        display_df['Mean_Accuracy'] = display_df['Mean_Accuracy'].round(4)
        display_df['Std_Accuracy'] = display_df['Std_Accuracy'].round(4)
        display_df['Improvement'] = display_df['Improvement'].round(4)
        
        print(display_df.to_string(index=False))
        
        # Save summary to CSV
        summary_path = Path(evaluator.report_dir) / f"final_evaluation_all_methods_summary_{data_name}.csv"
        display_df.to_csv(summary_path, index=False)
        print(f"\nSummary saved to: {summary_path}")

### 7. Key Insights

In [ ]:
if result_df is not None and len(baseline_acc) > 0:
    summary_sorted = summary.sort_values('Improvement_%', ascending=False)
    
    best_method = summary_sorted.iloc[0]
    worst_method = summary_sorted.iloc[-1]
    
    print("\n" + "="*100)
    print("KEY FINDINGS")
    print("="*100)
    print(f"\n✓ BEST PERFORMER:")
    print(f"  Method: {best_method['Method']} ({best_method['Model']})")
    print(f"  Accuracy: {best_method['Mean_Accuracy']:.4f} (±{best_method['Std_Accuracy']:.4f})")
    print(f"  Improvement over baseline: +{best_method['Improvement_%']:.2f}%")
    
    print(f"\n✗ WORST PERFORMER:")
    print(f"  Method: {worst_method['Method']} ({worst_method['Model']})")
    print(f"  Accuracy: {worst_method['Mean_Accuracy']:.4f} (±{worst_method['Std_Accuracy']:.4f})")
    print(f"  Change vs baseline: {worst_method['Improvement_%']:.2f}%")
    
    # Count which types improved
    improved_count = len(summary_sorted[summary_sorted['Improvement_%'] > 0])
    total_count = len(summary_sorted) - 1  # Exclude baseline
    
    print(f"\n📊 OVERALL STATISTICS:")
    print(f"  Methods evaluated: {len(summary_sorted) - 1}")
    print(f"  Methods improving baseline: {improved_count}/{total_count}")
    print(f"  Average improvement: {summary_sorted[summary_sorted['Method'] != 'None']['Improvement_%'].mean():.2f}%")